In [ ]:
# Cell 1: Import required libraries for external validation on the France BAAC 2024 dataset.

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Cell 2: Load the processed UK training dataset and France external-validation dataset.

uk_df = pd.read_csv(
    "../data/processed/uk_processed.csv"
)

france_df = pd.read_csv(
    "../data/processed/france_processed.csv"
)

print("UK shape:", uk_df.shape)
print("France shape:", france_df.shape)

print("\nUK columns:")
print(uk_df.columns.tolist())

print("\nFrance columns:")
print(france_df.columns.tolist())

In [ ]:
# Cell 3: Define the harmonized feature space shared by UK and France and verify the target column.

features = [
    "hour",
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common",
    "vehicle_count",
    "has_motorcycle",
    "has_heavy_vehicle",
    "has_public_transport",
    "has_two_wheeler"
]

target = "severity_class"

print("Number of common features:", len(features))

print("\nMissing features in UK:")
print([col for col in features if col not in uk_df.columns])

print("\nMissing features in France:")
print([col for col in features if col not in france_df.columns])

print("\nTarget available in UK:", target in uk_df.columns)
print("Target available in France:", target in france_df.columns)

print("\nUK target classes:")
print(uk_df[target].value_counts(dropna=False))

print("\nFrance target classes:")
print(france_df[target].value_counts(dropna=False))

In [ ]:
# Cell 4: Compare data types, missing values, and unique values of the harmonized features in UK and France.

for col in features:
    print("=" * 80)
    print(f"Feature: {col}")
    
    print("\nUK dtype:", uk_df[col].dtype)
    print("France dtype:", france_df[col].dtype)
    
    print("\nUK missing:", uk_df[col].isna().sum())
    print("France missing:", france_df[col].isna().sum())
    
    print("\nUK unique values:")
    print(sorted(uk_df[col].dropna().unique().tolist()))
    
    print("\nFrance unique values:")
    print(sorted(france_df[col].dropna().unique().tolist()))
    
    print()

In [ ]:
# Cell 5: Create the UK training set and France external-validation set using the harmonized features.

X_uk = uk_df[features].copy()
y_uk = uk_df[target].copy()

X_france = france_df[features].copy()
y_france = france_df[target].copy()

categorical_features = [
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common"
]

print("UK X shape:", X_uk.shape)
print("UK y shape:", y_uk.shape)

print("\nFrance X shape:", X_france.shape)
print("France y shape:", y_france.shape)

print("\nCategorical features:")
print(categorical_features)

print("\nUK missing values:", X_uk.isna().sum().sum())
print("France missing values:", X_france.isna().sum().sum())

In [ ]:
# Cell 6: Split the UK dataset into stratified training and internal test sets for comparison with France external validation.

X_train_uk, X_test_uk, y_train_uk, y_test_uk = train_test_split(
    X_uk,
    y_uk,
    test_size=0.20,
    random_state=42,
    stratify=y_uk
)

print("UK training set:", X_train_uk.shape)
print("UK internal test set:", X_test_uk.shape)

print("\nTraining class distribution:")
print(y_train_uk.value_counts())

print("\nInternal test class distribution:")
print(y_test_uk.value_counts())

print("\nFrance external validation set:", X_france.shape)
print("\nFrance class distribution:")
print(y_france.value_counts())

In [ ]:
# Cell 7: Train the CatBoost model exclusively on the UK training data using the harmonized feature space.

catboost_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False
)

catboost_model.fit(
    X_train_uk,
    y_train_uk,
    cat_features=categorical_features
)

print("CatBoost training completed successfully.")
print("Training samples:", len(X_train_uk))
print("Training features:", X_train_uk.shape[1])
print("France data used during training: No")

In [ ]:
# Cell 8: Evaluate the trained CatBoost model on the held-out UK test set for internal validation.

y_pred_uk = catboost_model.predict(X_test_uk).ravel()

uk_accuracy = accuracy_score(y_test_uk, y_pred_uk)
uk_precision_macro = precision_score(y_test_uk, y_pred_uk, average="macro", zero_division=0)
uk_recall_macro = recall_score(y_test_uk, y_pred_uk, average="macro", zero_division=0)
uk_f1_macro = f1_score(y_test_uk, y_pred_uk, average="macro", zero_division=0)

uk_precision_weighted = precision_score(y_test_uk, y_pred_uk, average="weighted", zero_division=0)
uk_recall_weighted = recall_score(y_test_uk, y_pred_uk, average="weighted", zero_division=0)
uk_f1_weighted = f1_score(y_test_uk, y_pred_uk, average="weighted", zero_division=0)

print("UK Internal Validation Results")
print("-" * 40)
print(f"Accuracy:           {uk_accuracy:.4f}")
print(f"Macro Precision:    {uk_precision_macro:.4f}")
print(f"Macro Recall:       {uk_recall_macro:.4f}")
print(f"Macro F1-score:     {uk_f1_macro:.4f}")
print(f"Weighted Precision: {uk_precision_weighted:.4f}")
print(f"Weighted Recall:    {uk_recall_weighted:.4f}")
print(f"Weighted F1-score:  {uk_f1_weighted:.4f}")

print("\nClassification Report:")
print(classification_report(y_test_uk, y_pred_uk, digits=4, zero_division=0))

In [ ]:
# Cell 9: Evaluate the UK-trained CatBoost model directly on the France dataset for true external validation.

y_pred_france = catboost_model.predict(X_france).ravel()

fr_accuracy = accuracy_score(y_france, y_pred_france)
fr_precision_macro = precision_score(
    y_france, y_pred_france, average="macro", zero_division=0
)
fr_recall_macro = recall_score(
    y_france, y_pred_france, average="macro", zero_division=0
)
fr_f1_macro = f1_score(
    y_france, y_pred_france, average="macro", zero_division=0
)

fr_precision_weighted = precision_score(
    y_france, y_pred_france, average="weighted", zero_division=0
)
fr_recall_weighted = recall_score(
    y_france, y_pred_france, average="weighted", zero_division=0
)
fr_f1_weighted = f1_score(
    y_france, y_pred_france, average="weighted", zero_division=0
)

print("France External Validation Results")
print("-" * 45)
print(f"Accuracy:           {fr_accuracy:.4f}")
print(f"Macro Precision:    {fr_precision_macro:.4f}")
print(f"Macro Recall:       {fr_recall_macro:.4f}")
print(f"Macro F1-score:     {fr_f1_macro:.4f}")
print(f"Weighted Precision: {fr_precision_weighted:.4f}")
print(f"Weighted Recall:    {fr_recall_weighted:.4f}")
print(f"Weighted F1-score:  {fr_f1_weighted:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_france,
        y_pred_france,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# Cell 10: Compare UK internal validation with France external validation and quantify the generalization performance drop.

comparison_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1-score",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1-score"
    ],
    "UK_Internal": [
        uk_accuracy,
        uk_precision_macro,
        uk_recall_macro,
        uk_f1_macro,
        uk_precision_weighted,
        uk_recall_weighted,
        uk_f1_weighted
    ],
    "France_External": [
        fr_accuracy,
        fr_precision_macro,
        fr_recall_macro,
        fr_f1_macro,
        fr_precision_weighted,
        fr_recall_weighted,
        fr_f1_weighted
    ]
})

comparison_df["Absolute_Change"] = (
    comparison_df["France_External"] - comparison_df["UK_Internal"]
)

comparison_df["Relative_Change_%"] = (
    comparison_df["Absolute_Change"] / comparison_df["UK_Internal"] * 100
)

comparison_df = comparison_df.round(4)

print("Internal vs External Validation Comparison")
print(comparison_df.to_string(index=False))

In [ ]:
# Cell 11: Compare class-specific precision, recall, and F1-score between UK internal validation and France external validation.

uk_class_report = classification_report(
    y_test_uk,
    y_pred_uk,
    labels=["Fatal", "Serious", "Slight"],
    output_dict=True,
    zero_division=0
)

fr_class_report = classification_report(
    y_france,
    y_pred_france,
    labels=["Fatal", "Serious", "Slight"],
    output_dict=True,
    zero_division=0
)

class_comparison = []

for cls in ["Fatal", "Serious", "Slight"]:
    class_comparison.append({
        "Class": cls,
        "UK_Precision": uk_class_report[cls]["precision"],
        "France_Precision": fr_class_report[cls]["precision"],
        "UK_Recall": uk_class_report[cls]["recall"],
        "France_Recall": fr_class_report[cls]["recall"],
        "UK_F1": uk_class_report[cls]["f1-score"],
        "France_F1": fr_class_report[cls]["f1-score"]
    })

class_comparison_df = pd.DataFrame(class_comparison).round(4)

print("Class-specific Internal vs External Validation")
print(class_comparison_df.to_string(index=False))

In [ ]:
# Cell 12: Generate confusion matrices for UK internal validation and France external validation using the same class order.

class_order = ["Fatal", "Serious", "Slight"]

cm_uk = confusion_matrix(
    y_test_uk,
    y_pred_uk,
    labels=class_order
)

cm_france = confusion_matrix(
    y_france,
    y_pred_france,
    labels=class_order
)

print("UK Internal Validation Confusion Matrix")
print(pd.DataFrame(
    cm_uk,
    index=[f"Actual_{c}" for c in class_order],
    columns=[f"Predicted_{c}" for c in class_order]
))

print("\nFrance External Validation Confusion Matrix")
print(pd.DataFrame(
    cm_france,
    index=[f"Actual_{c}" for c in class_order],
    columns=[f"Predicted_{c}" for c in class_order]
))

fig, ax = plt.subplots(figsize=(7, 6))

ConfusionMatrixDisplay(
    confusion_matrix=cm_france,
    display_labels=class_order
).plot(
    ax=ax,
    values_format="d",
    cmap="Blues",
    colorbar=False
)

ax.set_title("CatBoost External Validation — France BAAC 2024")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13: Calculate class weights from the UK training distribution to reduce bias toward the dominant Slight class.

class_counts = y_train_uk.value_counts()

total_samples = len(y_train_uk)
n_classes = len(class_counts)

class_weights = {
    cls: total_samples / (n_classes * count)
    for cls, count in class_counts.items()
}

print("UK training class counts:")
print(class_counts)

print("\nCalculated class weights:")
for cls, weight in class_weights.items():
    print(f"{cls}: {weight:.4f}")

In [ ]:
# Cell 14: Train a class-weighted CatBoost model on UK data to improve recognition of Serious and Fatal collisions.

weighted_catboost_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=class_weights
)

weighted_catboost_model.fit(
    X_train_uk,
    y_train_uk,
    cat_features=categorical_features
)

print("Weighted CatBoost training completed successfully.")
print("Class weights used:")
for cls, weight in class_weights.items():
    print(f"{cls}: {weight:.4f}")

In [ ]:
# Cell 15: Evaluate the class-weighted CatBoost model on the held-out UK test set and compare minority-class performance.

y_pred_uk_weighted = weighted_catboost_model.predict(X_test_uk).ravel()

uk_w_accuracy = accuracy_score(y_test_uk, y_pred_uk_weighted)
uk_w_precision_macro = precision_score(
    y_test_uk, y_pred_uk_weighted, average="macro", zero_division=0
)
uk_w_recall_macro = recall_score(
    y_test_uk, y_pred_uk_weighted, average="macro", zero_division=0
)
uk_w_f1_macro = f1_score(
    y_test_uk, y_pred_uk_weighted, average="macro", zero_division=0
)

uk_w_precision_weighted = precision_score(
    y_test_uk, y_pred_uk_weighted, average="weighted", zero_division=0
)
uk_w_recall_weighted = recall_score(
    y_test_uk, y_pred_uk_weighted, average="weighted", zero_division=0
)
uk_w_f1_weighted = f1_score(
    y_test_uk, y_pred_uk_weighted, average="weighted", zero_division=0
)

print("Weighted CatBoost — UK Internal Validation")
print("-" * 50)
print(f"Accuracy:           {uk_w_accuracy:.4f}")
print(f"Macro Precision:    {uk_w_precision_macro:.4f}")
print(f"Macro Recall:       {uk_w_recall_macro:.4f}")
print(f"Macro F1-score:     {uk_w_f1_macro:.4f}")
print(f"Weighted Precision: {uk_w_precision_weighted:.4f}")
print(f"Weighted Recall:    {uk_w_recall_weighted:.4f}")
print(f"Weighted F1-score:  {uk_w_f1_weighted:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_uk,
        y_pred_uk_weighted,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# Cell 16: Evaluate the class-weighted UK-trained CatBoost model on France for external validation.

y_pred_france_weighted = weighted_catboost_model.predict(X_france).ravel()

fr_w_accuracy = accuracy_score(
    y_france, y_pred_france_weighted
)

fr_w_precision_macro = precision_score(
    y_france, y_pred_france_weighted,
    average="macro",
    zero_division=0
)

fr_w_recall_macro = recall_score(
    y_france, y_pred_france_weighted,
    average="macro",
    zero_division=0
)

fr_w_f1_macro = f1_score(
    y_france, y_pred_france_weighted,
    average="macro",
    zero_division=0
)

fr_w_precision_weighted = precision_score(
    y_france, y_pred_france_weighted,
    average="weighted",
    zero_division=0
)

fr_w_recall_weighted = recall_score(
    y_france, y_pred_france_weighted,
    average="weighted",
    zero_division=0
)

fr_w_f1_weighted = f1_score(
    y_france, y_pred_france_weighted,
    average="weighted",
    zero_division=0
)

print("Weighted CatBoost — France External Validation")
print("-" * 55)

print(f"Accuracy:           {fr_w_accuracy:.4f}")
print(f"Macro Precision:    {fr_w_precision_macro:.4f}")
print(f"Macro Recall:       {fr_w_recall_macro:.4f}")
print(f"Macro F1-score:     {fr_w_f1_macro:.4f}")
print(f"Weighted Precision: {fr_w_precision_weighted:.4f}")
print(f"Weighted Recall:    {fr_w_recall_weighted:.4f}")
print(f"Weighted F1-score:  {fr_w_f1_weighted:.4f}")

print("\nClassification Report:")

print(
    classification_report(
        y_france,
        y_pred_france_weighted,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# Cell 17: Compare baseline and weighted CatBoost performance on UK and France in one summary table.

summary_results = pd.DataFrame({
    "Model": [
        "Baseline CatBoost",
        "Weighted CatBoost"
    ],
    "UK_Accuracy": [
        uk_accuracy,
        uk_w_accuracy
    ],
    "UK_Macro_F1": [
        uk_f1_macro,
        uk_w_f1_macro
    ],
    "UK_Macro_Recall": [
        uk_recall_macro,
        uk_w_recall_macro
    ],
    "France_Accuracy": [
        fr_accuracy,
        fr_w_accuracy
    ],
    "France_Macro_F1": [
        fr_f1_macro,
        fr_w_f1_macro
    ],
    "France_Macro_Recall": [
        fr_recall_macro,
        fr_w_recall_macro
    ]
})

summary_results = summary_results.round(4)

print("Baseline vs Weighted CatBoost")
print(summary_results.to_string(index=False))

In [ ]:
# Cell 18: Generate moderated class weights by applying different exponent values to the balanced weights.

alpha_values = [0.0, 0.25, 0.5, 0.75, 1.0]

for alpha in alpha_values:
    adjusted_weights = {
        cls: weight ** alpha
        for cls, weight in class_weights.items()
    }

    print(f"\nAlpha = {alpha}")
    for cls, weight in adjusted_weights.items():
        print(f"{cls}: {weight:.4f}")

In [ ]:
# Cell 19: Train and evaluate CatBoost models with different power-smoothed class-weight settings using only the UK internal validation set.

alpha_values = [0.0, 0.25, 0.5, 0.75, 1.0]

alpha_results = []

for alpha in alpha_values:
    
    adjusted_weights = {
        cls: weight ** alpha
        for cls, weight in class_weights.items()
    }
    
    model = CatBoostClassifier(
        iterations=500,
        depth=8,
        learning_rate=0.05,
        loss_function="MultiClass",
        random_seed=42,
        verbose=False,
        class_weights=adjusted_weights
    )
    
    model.fit(
        X_train_uk,
        y_train_uk,
        cat_features=categorical_features
    )
    
    y_pred = model.predict(X_test_uk).ravel()
    
    report = classification_report(
        y_test_uk,
        y_pred,
        labels=["Fatal", "Serious", "Slight"],
        output_dict=True,
        zero_division=0
    )
    
    alpha_results.append({
        "Alpha": alpha,
        "Accuracy": accuracy_score(y_test_uk, y_pred),
        "Macro_Precision": precision_score(
            y_test_uk, y_pred, average="macro", zero_division=0
        ),
        "Macro_Recall": recall_score(
            y_test_uk, y_pred, average="macro", zero_division=0
        ),
        "Macro_F1": f1_score(
            y_test_uk, y_pred, average="macro", zero_division=0
        ),
        "Fatal_Precision": report["Fatal"]["precision"],
        "Fatal_Recall": report["Fatal"]["recall"],
        "Fatal_F1": report["Fatal"]["f1-score"],
        "Serious_Precision": report["Serious"]["precision"],
        "Serious_Recall": report["Serious"]["recall"],
        "Serious_F1": report["Serious"]["f1-score"],
        "Slight_Recall": report["Slight"]["recall"]
    })

alpha_results_df = pd.DataFrame(alpha_results).round(4)

print("Power-Smoothed Class Weight Comparison — UK Internal Validation")
print(alpha_results_df.to_string(index=False))

In [ ]:
# Cell 20: Train the final cost-sensitive CatBoost model using alpha=0.75 selected by the highest UK internal Macro-F1.

selected_alpha = 0.75

final_class_weights = {
    cls: weight ** selected_alpha
    for cls, weight in class_weights.items()
}

print("Selected alpha:", selected_alpha)
print("\nFinal class weights:")

for cls, weight in final_class_weights.items():
    print(f"{cls}: {weight:.4f}")

final_catboost_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=final_class_weights
)

final_catboost_model.fit(
    X_train_uk,
    y_train_uk,
    cat_features=categorical_features
)

print("\nFinal cost-sensitive CatBoost model trained successfully.")
print("Model selection criterion: Highest Macro-F1 on UK internal validation.")
print("France data used for alpha selection: No")
print("France data used for model training: No")

In [ ]:
# Cell 21: Evaluate the final alpha=0.75 cost-sensitive CatBoost model on France as the untouched external-validation dataset.

y_pred_france_final = final_catboost_model.predict(X_france).ravel()

fr_final_accuracy = accuracy_score(
    y_france, y_pred_france_final
)

fr_final_precision_macro = precision_score(
    y_france, y_pred_france_final,
    average="macro",
    zero_division=0
)

fr_final_recall_macro = recall_score(
    y_france, y_pred_france_final,
    average="macro",
    zero_division=0
)

fr_final_f1_macro = f1_score(
    y_france, y_pred_france_final,
    average="macro",
    zero_division=0
)

fr_final_precision_weighted = precision_score(
    y_france, y_pred_france_final,
    average="weighted",
    zero_division=0
)

fr_final_recall_weighted = recall_score(
    y_france, y_pred_france_final,
    average="weighted",
    zero_division=0
)

fr_final_f1_weighted = f1_score(
    y_france, y_pred_france_final,
    average="weighted",
    zero_division=0
)

print("Final Cost-Sensitive CatBoost — France External Validation")
print("-" * 65)

print(f"Accuracy:           {fr_final_accuracy:.4f}")
print(f"Macro Precision:    {fr_final_precision_macro:.4f}")
print(f"Macro Recall:       {fr_final_recall_macro:.4f}")
print(f"Macro F1-score:     {fr_final_f1_macro:.4f}")
print(f"Weighted Precision: {fr_final_precision_weighted:.4f}")
print(f"Weighted Recall:    {fr_final_recall_weighted:.4f}")
print(f"Weighted F1-score:  {fr_final_f1_weighted:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_france,
        y_pred_france_final,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

In [ ]:
# Cell 22: Compare baseline, fully balanced, and final alpha=0.75 CatBoost models on France external validation.

final_comparison_df = pd.DataFrame({
    "Model": [
        "Baseline CatBoost",
        "Fully Balanced CatBoost (alpha=1.0)",
        "Final Cost-Sensitive CatBoost (alpha=0.75)"
    ],
    "Accuracy": [
        fr_accuracy,
        fr_w_accuracy,
        fr_final_accuracy
    ],
    "Macro_Precision": [
        fr_precision_macro,
        fr_w_precision_macro,
        fr_final_precision_macro
    ],
    "Macro_Recall": [
        fr_recall_macro,
        fr_w_recall_macro,
        fr_final_recall_macro
    ],
    "Macro_F1": [
        fr_f1_macro,
        fr_w_f1_macro,
        fr_final_f1_macro
    ],
    "Weighted_F1": [
        fr_f1_weighted,
        fr_w_f1_weighted,
        fr_final_f1_weighted
    ]
}).round(4)

print("France External Validation — Model Comparison")
print(final_comparison_df.to_string(index=False))

In [ ]:
# Cell 23: Generate and visualize the confusion matrix of the final alpha=0.75 CatBoost model on France external validation.

class_order = ["Fatal", "Serious", "Slight"]

cm_france_final = confusion_matrix(
    y_france,
    y_pred_france_final,
    labels=class_order
)

cm_france_final_df = pd.DataFrame(
    cm_france_final,
    index=[f"Actual_{c}" for c in class_order],
    columns=[f"Predicted_{c}" for c in class_order]
)

print("Final Cost-Sensitive CatBoost — France Confusion Matrix")
print(cm_france_final_df)

fig, ax = plt.subplots(figsize=(7, 6))

ConfusionMatrixDisplay(
    confusion_matrix=cm_france_final,
    display_labels=class_order
).plot(
    ax=ax,
    values_format="d",
    cmap="Blues",
    colorbar=False
)

ax.set_title(
    "Final Cost-Sensitive CatBoost — France External Validation"
)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 24: Generate a row-normalized confusion matrix to show class-specific prediction percentages for the final France external validation.

cm_france_final_normalized = (
    cm_france_final.astype(float) /
    cm_france_final.sum(axis=1, keepdims=True)
)

cm_france_final_percent = cm_france_final_normalized * 100

cm_france_final_percent_df = pd.DataFrame(
    cm_france_final_percent,
    index=[f"Actual_{c}" for c in class_order],
    columns=[f"Predicted_{c}" for c in class_order]
).round(2)

print("Normalized France External Validation Confusion Matrix (%)")
print(cm_france_final_percent_df)

fig, ax = plt.subplots(figsize=(7, 6))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_france_final_normalized,
    display_labels=class_order
)

disp.plot(
    ax=ax,
    values_format=".2f",
    cmap="Blues",
    colorbar=False
)

ax.set_title(
    "Normalized Confusion Matrix — France External Validation"
)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 24: Inspect final-model prediction distributions by true severity class to diagnose systematic underestimation of severe collisions.

diagnostic_df = pd.DataFrame({
    "Actual": y_france.values,
    "Predicted": y_pred_france_final
})

prediction_distribution = pd.crosstab(
    diagnostic_df["Actual"],
    diagnostic_df["Predicted"],
    normalize="index"
) * 100

prediction_distribution = prediction_distribution.reindex(
    index=["Fatal", "Serious", "Slight"],
    columns=["Fatal", "Serious", "Slight"],
    fill_value=0
).round(2)

print("France Prediction Distribution by Actual Severity (%)")
print(prediction_distribution)

print("\nOverall predicted class distribution:")
print(
    diagnostic_df["Predicted"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nActual class distribution:")
print(
    diagnostic_df["Actual"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
# Cell 25: Inspect candidate raw variables in UK and France that may support additional cross-country harmonized severity predictors.

uk_candidates = [
    "speed_limit",
    "road_type",
    "urban_or_rural_area",
    "number_of_vehicles",
    "number_of_casualties",
    "first_road_class",
    "junction_detail",
    "junction_control",
    "weather_conditions",
    "light_conditions",
    "road_surface_conditions"
]

france_candidates = [
    "agg",
    "int",
    "col",
    "lum",
    "atm",
    "vehicle_count",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common"
]

print("UK CANDIDATE VARIABLES")
print("=" * 80)

for col in uk_candidates:
    if col in uk_df.columns:
        print(f"\n{col}")
        print("dtype:", uk_df[col].dtype)
        print("missing:", uk_df[col].isna().sum())
        print("unique:", uk_df[col].nunique(dropna=True))
        print(uk_df[col].value_counts(dropna=False).head(15))

print("\n\nFRANCE CANDIDATE VARIABLES")
print("=" * 80)

for col in france_candidates:
    if col in france_df.columns:
        print(f"\n{col}")
        print("dtype:", france_df[col].dtype)
        print("missing:", france_df[col].isna().sum())
        print("unique:", france_df[col].nunique(dropna=True))
        print(france_df[col].value_counts(dropna=False).head(15))

In [ ]:
# Cell 25: Create hierarchical severity targets for two-stage classification without using France during model development.

y_uk_stage1 = y_uk.map({
    "Slight": "Slight",
    "Serious": "Severe",
    "Fatal": "Severe"
})

y_france_stage1 = y_france.map({
    "Slight": "Slight",
    "Serious": "Severe",
    "Fatal": "Severe"
})

print("UK — Stage 1 target distribution:")
print(y_uk_stage1.value_counts())

print("\nUK — Stage 1 percentages:")
print(
    y_uk_stage1.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nUK — Stage 2 severe cases:")
print(
    y_uk[y_uk.isin(["Serious", "Fatal"])]
    .value_counts()
)

print("\nFrance — Stage 1 target distribution:")
print(y_france_stage1.value_counts())

print("\nFrance — Stage 1 percentages:")
print(
    y_france_stage1.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nFrance — Stage 2 severe cases:")
print(
    y_france[y_france.isin(["Serious", "Fatal"])]
    .value_counts()
)

In [ ]:
# Cell 26: Create a stratified 80/20 UK split for Stage 1 (Slight vs Severe) while keeping France completely external.

X_train_stage1, X_test_stage1, y_train_stage1, y_test_stage1 = train_test_split(
    X_uk,
    y_uk_stage1,
    test_size=0.20,
    random_state=42,
    stratify=y_uk_stage1
)

print("Stage 1 — UK training set:", X_train_stage1.shape)
print("Stage 1 — UK internal test set:", X_test_stage1.shape)

print("\nTraining distribution:")
print(y_train_stage1.value_counts())

print("\nTraining percentages:")
print(
    y_train_stage1.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nInternal test distribution:")
print(y_test_stage1.value_counts())

print("\nInternal test percentages:")
print(
    y_test_stage1.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nFrance remains external:", X_france.shape)
print("France used for training: No")

In [ ]:
# Cell 27: Train a baseline binary CatBoost model on UK data to test whether Severe collisions can be distinguished from Slight collisions.

stage1_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="Logloss",
    random_seed=42,
    verbose=False
)

stage1_model.fit(
    X_train_stage1,
    y_train_stage1,
    cat_features=categorical_features
)

print("Stage 1 CatBoost training completed successfully.")
print("Target: Slight vs Severe")
print("Training samples:", X_train_stage1.shape[0])
print("Training features:", X_train_stage1.shape[1])
print("France data used during training: No")

In [ ]:
# Cell 28: Evaluate the Stage 1 binary CatBoost model on the UK internal test set (Slight vs Severe).

y_pred_stage1_uk = stage1_model.predict(X_test_stage1).ravel()

stage1_accuracy = accuracy_score(
    y_test_stage1,
    y_pred_stage1_uk
)

stage1_precision_macro = precision_score(
    y_test_stage1,
    y_pred_stage1_uk,
    average="macro",
    zero_division=0
)

stage1_recall_macro = recall_score(
    y_test_stage1,
    y_pred_stage1_uk,
    average="macro",
    zero_division=0
)

stage1_f1_macro = f1_score(
    y_test_stage1,
    y_pred_stage1_uk,
    average="macro",
    zero_division=0
)

print("Stage 1 CatBoost — UK Internal Validation")
print("-" * 55)

print(f"Accuracy:        {stage1_accuracy:.4f}")
print(f"Macro Precision: {stage1_precision_macro:.4f}")
print(f"Macro Recall:    {stage1_recall_macro:.4f}")
print(f"Macro F1-score:  {stage1_f1_macro:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_stage1,
        y_pred_stage1_uk,
        labels=["Severe", "Slight"],
        digits=4,
        zero_division=0
    )
)

cm_stage1_uk = confusion_matrix(
    y_test_stage1,
    y_pred_stage1_uk,
    labels=["Severe", "Slight"]
)

cm_stage1_uk_df = pd.DataFrame(
    cm_stage1_uk,
    index=["Actual_Severe", "Actual_Slight"],
    columns=["Predicted_Severe", "Predicted_Slight"]
)

print("\nConfusion Matrix:")
print(cm_stage1_uk_df)

print("\nRow-normalized Confusion Matrix (%):")

cm_stage1_uk_pct = (
    cm_stage1_uk.astype(float)
    / cm_stage1_uk.sum(axis=1, keepdims=True)
    * 100
)

print(
    pd.DataFrame(
        cm_stage1_uk_pct,
        index=["Actual_Severe", "Actual_Slight"],
        columns=["Predicted_Severe", "Predicted_Slight"]
    ).round(2)
)

In [ ]:
# Cell 30: Reconstruct harmonized vehicle categories from the original UK and France vehicle-level files to study collision configurations.

uk_vehicles_raw = pd.read_csv(
    "../data/raw/UK/dft-road-casualty-statistics-vehicle-2025.csv",
    low_memory=False
)

fr_vehicles_raw = pd.read_csv(
    "../data/raw/France/vehicules-2024.csv",
    sep=";",
    low_memory=False
)

# همان Mapping استفاده‌شده در Notebook 02
uk_vehicle_category_map = {
    1: "two_wheeler",
    2: "motorcycle",
    3: "motorcycle",
    4: "motorcycle",
    5: "motorcycle",
    8: "taxi",
    9: "car",
    10: "public_transport",
    11: "public_transport",
    16: "special_vehicle",
    17: "special_vehicle",
    18: "special_vehicle",
    19: "light_commercial",
    20: "heavy_vehicle",
    21: "heavy_vehicle",
    22: "two_wheeler",
    23: "motorcycle",
    90: "other",
    97: "motorcycle",
    98: "heavy_vehicle",
    99: "unknown",
    103: "motorcycle",
    104: "motorcycle",
    105: "motorcycle",
    106: "motorcycle",
    108: "taxi",
    109: "car",
    110: "special_vehicle",
    113: "heavy_vehicle",
    -1: "unknown"
}

fr_vehicle_category_map = {
    1: "two_wheeler",
    2: "motorcycle",
    30: "motorcycle",
    31: "motorcycle",
    32: "motorcycle",
    33: "motorcycle",
    34: "motorcycle",
    7: "car",
    10: "car",
    13: "heavy_vehicle",
    14: "heavy_vehicle",
    15: "heavy_vehicle",
    17: "heavy_vehicle",
    20: "heavy_vehicle",
    21: "heavy_vehicle",
    37: "public_transport",
    38: "public_transport",
    60: "two_wheeler",
    80: "two_wheeler",
    50: "other",
    99: "unknown",
    0: "unknown",
    -1: "unknown"
}

uk_vehicles_raw["vehicle_category"] = (
    uk_vehicles_raw["vehicle_type"]
    .map(uk_vehicle_category_map)
    .fillna("other")
)

fr_vehicles_raw["vehicle_category"] = (
    fr_vehicles_raw["catv"]
    .map(fr_vehicle_category_map)
    .fillna("other")
)

print("UK vehicle-category distribution:")
print(uk_vehicles_raw["vehicle_category"].value_counts())

print("\nFrance vehicle-category distribution:")
print(fr_vehicles_raw["vehicle_category"].value_counts())

print("\nUK vehicle records:", len(uk_vehicles_raw))
print("France vehicle records:", len(fr_vehicles_raw))

In [ ]:
# Cell 31: Aggregate vehicle-level records to collision-level vehicle composition for UK and France.

uk_collision_id = "collision_index"
fr_collision_id = "Num_Acc"

# Count each harmonized vehicle category within every UK collision
uk_vehicle_counts = (
    uk_vehicles_raw
    .groupby([uk_collision_id, "vehicle_category"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Count each harmonized vehicle category within every France collision
fr_vehicle_counts = (
    fr_vehicles_raw
    .groupby([fr_collision_id, "vehicle_category"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Ensure the main categories exist in both tables
vehicle_groups = [
    "car",
    "motorcycle",
    "two_wheeler",
    "heavy_vehicle",
    "public_transport",
    "light_commercial",
    "taxi",
    "special_vehicle",
    "other",
    "unknown"
]

for col in vehicle_groups:
    if col not in uk_vehicle_counts.columns:
        uk_vehicle_counts[col] = 0
    if col not in fr_vehicle_counts.columns:
        fr_vehicle_counts[col] = 0

uk_vehicle_counts["total_vehicles_from_file"] = (
    uk_vehicle_counts[vehicle_groups].sum(axis=1)
)

fr_vehicle_counts["total_vehicles_from_file"] = (
    fr_vehicle_counts[vehicle_groups].sum(axis=1)
)

print("UK collision-level vehicle table:", uk_vehicle_counts.shape)
print("France collision-level vehicle table:", fr_vehicle_counts.shape)

print("\nUK example:")
print(uk_vehicle_counts.head())

print("\nFrance example:")
print(fr_vehicle_counts.head())

print("\nUK number of vehicles per collision:")
print(uk_vehicle_counts["total_vehicles_from_file"].value_counts().sort_index().head(10))

print("\nFrance number of vehicles per collision:")
print(fr_vehicle_counts["total_vehicles_from_file"].value_counts().sort_index().head(10))

In [ ]:
# Cell 32: Create interpretable collision-configuration labels from the vehicle composition in UK and France.

def assign_collision_configuration(row):
    total = row["total_vehicles_from_file"]

    # Single-vehicle crashes
    if total == 1:
        if row["car"] == 1:
            return "Single_Car"
        if row["motorcycle"] == 1:
            return "Single_Motorcycle"
        if row["two_wheeler"] == 1:
            return "Single_TwoWheeler"
        if row["heavy_vehicle"] == 1:
            return "Single_HeavyVehicle"
        if row["public_transport"] == 1:
            return "Single_PublicTransport"
        return "Single_Other"

    # Two-vehicle crashes
    if total == 2:
        if row["car"] == 2:
            return "Car-Car"

        if row["car"] == 1 and row["motorcycle"] == 1:
            return "Car-Motorcycle"

        if row["car"] == 1 and row["two_wheeler"] == 1:
            return "Car-TwoWheeler"

        if row["car"] == 1 and row["heavy_vehicle"] == 1:
            return "Car-HeavyVehicle"

        if row["car"] == 1 and row["public_transport"] == 1:
            return "Car-PublicTransport"

        if row["motorcycle"] == 2:
            return "Motorcycle-Motorcycle"

        if row["motorcycle"] == 1 and row["heavy_vehicle"] == 1:
            return "Motorcycle-HeavyVehicle"

        if row["two_wheeler"] == 1 and row["heavy_vehicle"] == 1:
            return "TwoWheeler-HeavyVehicle"

        if row["heavy_vehicle"] == 2:
            return "HeavyVehicle-HeavyVehicle"

        return "TwoVehicle_Other"

    # Three or more vehicles
    if total >= 3:
        return "MultiVehicle_3plus"

    return "Unknown"


uk_vehicle_counts["collision_configuration"] = (
    uk_vehicle_counts.apply(assign_collision_configuration, axis=1)
)

fr_vehicle_counts["collision_configuration"] = (
    fr_vehicle_counts.apply(assign_collision_configuration, axis=1)
)

print("UK collision configurations:")
print(uk_vehicle_counts["collision_configuration"].value_counts())

print("\nFrance collision configurations:")
print(fr_vehicle_counts["collision_configuration"].value_counts())

print("\nConfigurations present in both countries:")

common_configs = sorted(
    set(uk_vehicle_counts["collision_configuration"].unique())
    &
    set(fr_vehicle_counts["collision_configuration"].unique())
)

print(common_configs)

In [ ]:
# Cell 33: Link collision configurations to severity and calculate the observed Slight/Serious/Fatal distribution for each configuration in UK and France.

# Attach collision configuration to the accident-level datasets
uk_config_severity = uk_df[
    ["collision_index", "severity_class"]
].merge(
    uk_vehicle_counts[
        ["collision_index", "collision_configuration"]
    ],
    on="collision_index",
    how="inner"
)

fr_config_severity = france_df[
    ["Num_Acc", "severity_class"]
].merge(
    fr_vehicle_counts[
        ["Num_Acc", "collision_configuration"]
    ],
    on="Num_Acc",
    how="inner"
)

severity_order = ["Slight", "Serious", "Fatal"]

# UK counts
uk_config_counts = pd.crosstab(
    uk_config_severity["collision_configuration"],
    uk_config_severity["severity_class"]
).reindex(columns=severity_order, fill_value=0)

uk_config_counts["Total"] = uk_config_counts.sum(axis=1)

# UK percentages
uk_config_pct = (
    uk_config_counts[severity_order]
    .div(uk_config_counts["Total"], axis=0)
    .mul(100)
    .round(2)
)

uk_config_pct["Total"] = uk_config_counts["Total"]


# France counts
fr_config_counts = pd.crosstab(
    fr_config_severity["collision_configuration"],
    fr_config_severity["severity_class"]
).reindex(columns=severity_order, fill_value=0)

fr_config_counts["Total"] = fr_config_counts.sum(axis=1)

# France percentages
fr_config_pct = (
    fr_config_counts[severity_order]
    .div(fr_config_counts["Total"], axis=0)
    .mul(100)
    .round(2)
)

fr_config_pct["Total"] = fr_config_counts["Total"]


print("UK — Severity Distribution by Collision Configuration (%)")
print("=" * 85)
print(
    uk_config_pct
    .sort_values("Total", ascending=False)
    .to_string()
)

print("\n\nFrance — Severity Distribution by Collision Configuration (%)")
print("=" * 85)
print(
    fr_config_pct
    .sort_values("Total", ascending=False)
    .to_string()
)

In [ ]:
# Cell 34: Add collision configuration as a new harmonized predictor and prepare the expanded UK and France feature sets.

# Add collision configuration to UK accident-level data
uk_expanded = uk_df.merge(
    uk_vehicle_counts[
        ["collision_index", "collision_configuration"]
    ],
    on="collision_index",
    how="left"
)

# Add collision configuration to France accident-level data
france_expanded = france_df.merge(
    fr_vehicle_counts[
        ["Num_Acc", "collision_configuration"]
    ],
    on="Num_Acc",
    how="left"
)

# Original 11 common features + collision configuration
expanded_features = features + ["collision_configuration"]

X_uk_expanded = uk_expanded[expanded_features].copy()
y_uk_expanded = uk_expanded["severity_class"].copy()

X_france_expanded = france_expanded[expanded_features].copy()
y_france_expanded = france_expanded["severity_class"].copy()

expanded_categorical_features = categorical_features + [
    "collision_configuration"
]

print("Number of original features:", len(features))
print("Number of expanded features:", len(expanded_features))

print("\nExpanded features:")
print(expanded_features)

print("\nUK expanded shape:", X_uk_expanded.shape)
print("France expanded shape:", X_france_expanded.shape)

print("\nUK missing collision configurations:")
print(X_uk_expanded["collision_configuration"].isna().sum())

print("\nFrance missing collision configurations:")
print(X_france_expanded["collision_configuration"].isna().sum())

print("\nCategorical features:")
print(expanded_categorical_features)

In [ ]:
# Cell 35: Train and evaluate a baseline CatBoost model with the new collision_configuration feature using the same 80/20 stratified UK split.

X_train_expanded, X_test_expanded, y_train_expanded, y_test_expanded = train_test_split(
    X_uk_expanded,
    y_uk_expanded,
    test_size=0.20,
    random_state=42,
    stratify=y_uk_expanded
)

expanded_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False
)

expanded_model.fit(
    X_train_expanded,
    y_train_expanded,
    cat_features=expanded_categorical_features
)

y_pred_expanded_uk = expanded_model.predict(X_test_expanded).ravel()

expanded_uk_accuracy = accuracy_score(
    y_test_expanded, y_pred_expanded_uk
)

expanded_uk_macro_precision = precision_score(
    y_test_expanded,
    y_pred_expanded_uk,
    average="macro",
    zero_division=0
)

expanded_uk_macro_recall = recall_score(
    y_test_expanded,
    y_pred_expanded_uk,
    average="macro",
    zero_division=0
)

expanded_uk_macro_f1 = f1_score(
    y_test_expanded,
    y_pred_expanded_uk,
    average="macro",
    zero_division=0
)

expanded_uk_report = classification_report(
    y_test_expanded,
    y_pred_expanded_uk,
    labels=["Fatal", "Serious", "Slight"],
    output_dict=True,
    zero_division=0
)

print("Expanded 12-Feature CatBoost — UK Internal Validation")
print("-" * 60)

print(f"Accuracy:        {expanded_uk_accuracy:.4f}")
print(f"Macro Precision: {expanded_uk_macro_precision:.4f}")
print(f"Macro Recall:    {expanded_uk_macro_recall:.4f}")
print(f"Macro F1-score:  {expanded_uk_macro_f1:.4f}")

print("\nClass-specific results:")
print(
    classification_report(
        y_test_expanded,
        y_pred_expanded_uk,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

print("\nComparison with original 11-feature baseline:")
print(f"11-feature Accuracy: {uk_accuracy:.4f}")
print(f"12-feature Accuracy: {expanded_uk_accuracy:.4f}")

print(f"\n11-feature Macro F1: {uk_f1_macro:.4f}")
print(f"12-feature Macro F1: {expanded_uk_macro_f1:.4f}")

print(f"\n11-feature Fatal Recall: {uk_class_report['Fatal']['recall']:.4f}")
print(f"12-feature Fatal Recall: {expanded_uk_report['Fatal']['recall']:.4f}")

print(f"\n11-feature Serious Recall: {uk_class_report['Serious']['recall']:.4f}")
print(f"12-feature Serious Recall: {expanded_uk_report['Serious']['recall']:.4f}")

In [ ]:
# Cell 36: Evaluate the 12-feature CatBoost model using the previously selected alpha=0.75 class-weighting strategy on UK internal validation.

expanded_class_weights = {
    cls: weight ** 0.75
    for cls, weight in class_weights.items()
}

expanded_weighted_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False,
    class_weights=expanded_class_weights
)

expanded_weighted_model.fit(
    X_train_expanded,
    y_train_expanded,
    cat_features=expanded_categorical_features
)

y_pred_expanded_weighted_uk = (
    expanded_weighted_model
    .predict(X_test_expanded)
    .ravel()
)

expanded_w_accuracy = accuracy_score(
    y_test_expanded,
    y_pred_expanded_weighted_uk
)

expanded_w_macro_precision = precision_score(
    y_test_expanded,
    y_pred_expanded_weighted_uk,
    average="macro",
    zero_division=0
)

expanded_w_macro_recall = recall_score(
    y_test_expanded,
    y_pred_expanded_weighted_uk,
    average="macro",
    zero_division=0
)

expanded_w_macro_f1 = f1_score(
    y_test_expanded,
    y_pred_expanded_weighted_uk,
    average="macro",
    zero_division=0
)

expanded_w_report = classification_report(
    y_test_expanded,
    y_pred_expanded_weighted_uk,
    labels=["Fatal", "Serious", "Slight"],
    output_dict=True,
    zero_division=0
)

print("12-Feature Cost-Sensitive CatBoost (alpha=0.75) — UK")
print("-" * 65)

print(f"Accuracy:        {expanded_w_accuracy:.4f}")
print(f"Macro Precision: {expanded_w_macro_precision:.4f}")
print(f"Macro Recall:    {expanded_w_macro_recall:.4f}")
print(f"Macro F1-score:  {expanded_w_macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_expanded,
        y_pred_expanded_weighted_uk,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

print("\n11 vs 12 Features — Both with alpha=0.75")
print("-" * 55)

print(f"11-feature Accuracy:       0.6634")
print(f"12-feature Accuracy:       {expanded_w_accuracy:.4f}")

print(f"\n11-feature Macro F1:       0.4014")
print(f"12-feature Macro F1:       {expanded_w_macro_f1:.4f}")

print(f"\n11-feature Fatal Recall:   0.1649")
print(f"12-feature Fatal Recall:   {expanded_w_report['Fatal']['recall']:.4f}")

print(f"\n11-feature Serious Recall: 0.3069")
print(f"12-feature Serious Recall: {expanded_w_report['Serious']['recall']:.4f}")

print(f"\n11-feature Slight Recall:  0.7930")
print(f"12-feature Slight Recall:  {expanded_w_report['Slight']['recall']:.4f}")

In [ ]:
# Cell 37: Create a controlled UK training set by undersampling the majority classes while preserving all Fatal cases.

train_resample_df = X_train_uk.copy()
train_resample_df["severity_class"] = y_train_uk.values

fatal_train = train_resample_df[
    train_resample_df["severity_class"] == "Fatal"
]

serious_train = train_resample_df[
    train_resample_df["severity_class"] == "Serious"
]

slight_train = train_resample_df[
    train_resample_df["severity_class"] == "Slight"
]

# Keep all Fatal cases and reduce the two majority classes
slight_sample = slight_train.sample(
    n=20000,
    random_state=42
)

serious_sample = serious_train.sample(
    n=20000,
    random_state=42
)

train_resampled = pd.concat(
    [
        slight_sample,
        serious_sample,
        fatal_train
    ],
    axis=0
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

X_train_resampled = train_resampled[features].copy()
y_train_resampled = train_resampled["severity_class"].copy()

print("Original UK training distribution:")
print(y_train_uk.value_counts())

print("\nControlled-resampling training distribution:")
print(y_train_resampled.value_counts())

print("\nControlled-resampling percentages:")
print(
    y_train_resampled
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nOriginal training size:", len(y_train_uk))
print("Resampled training size:", len(y_train_resampled))

print("\nUK internal test set remains unchanged:", X_test_uk.shape)
print("France data used: No")

In [ ]:
# Cell 38: Train and evaluate CatBoost on the controlled-resampling UK training set without additional class weights.

resampled_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    random_seed=42,
    verbose=False
)

resampled_model.fit(
    X_train_resampled,
    y_train_resampled,
    cat_features=categorical_features
)

y_pred_resampled_uk = resampled_model.predict(X_test_uk).ravel()

resampled_accuracy = accuracy_score(
    y_test_uk,
    y_pred_resampled_uk
)

resampled_macro_precision = precision_score(
    y_test_uk,
    y_pred_resampled_uk,
    average="macro",
    zero_division=0
)

resampled_macro_recall = recall_score(
    y_test_uk,
    y_pred_resampled_uk,
    average="macro",
    zero_division=0
)

resampled_macro_f1 = f1_score(
    y_test_uk,
    y_pred_resampled_uk,
    average="macro",
    zero_division=0
)

resampled_report = classification_report(
    y_test_uk,
    y_pred_resampled_uk,
    labels=["Fatal", "Serious", "Slight"],
    output_dict=True,
    zero_division=0
)

print("Controlled-Resampling CatBoost — UK Internal Validation")
print("-" * 65)

print(f"Accuracy:        {resampled_accuracy:.4f}")
print(f"Macro Precision: {resampled_macro_precision:.4f}")
print(f"Macro Recall:    {resampled_macro_recall:.4f}")
print(f"Macro F1-score:  {resampled_macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_uk,
        y_pred_resampled_uk,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

print("\nComparison with previous models")
print("-" * 45)

print(f"Baseline Macro F1:              {uk_f1_macro:.4f}")
print(f"Alpha=0.75 Macro F1:            0.4014")
print(f"Controlled-resampling Macro F1: {resampled_macro_f1:.4f}")

print(f"\nBaseline Fatal Recall:              {uk_class_report['Fatal']['recall']:.4f}")
print(f"Alpha=0.75 Fatal Recall:            0.1649")
print(f"Controlled-resampling Fatal Recall: {resampled_report['Fatal']['recall']:.4f}")

print(f"\nBaseline Serious Recall:              {uk_class_report['Serious']['recall']:.4f}")
print(f"Alpha=0.75 Serious Recall:            0.3069")
print(f"Controlled-resampling Serious Recall: {resampled_report['Serious']['recall']:.4f}")

In [ ]:
# Cell 39: Train a multinomial Logistic Regression baseline on the same 11-feature UK training set for a fair comparison with ensemble models.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

numeric_features = [
    "hour",
    "vehicle_count",
    "has_motorcycle",
    "has_heavy_vehicle",
    "has_public_transport",
    "has_two_wheeler"
]

categorical_features_lr = [
    "day_common",
    "weather_common",
    "light_common",
    "surface_common",
    "junction_common"
]

preprocessor_lr = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features_lr
        )
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_lr),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(
    X_train_uk,
    y_train_uk
)

y_pred_lr_uk = logistic_model.predict(
    X_test_uk
)

print("Logistic Regression — UK Internal Validation")
print("-" * 55)

print(
    classification_report(
        y_test_uk,
        y_pred_lr_uk,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

print(
    "Accuracy:",
    round(
        accuracy_score(
            y_test_uk,
            y_pred_lr_uk
        ),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_test_uk,
            y_pred_lr_uk,
            average="macro",
            zero_division=0
        ),
        4
    )
)

In [ ]:
# Cell 40: Train the XGBoost classifier on the same 11 harmonized UK features for direct comparison with CatBoost and Logistic Regression.

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# Encode target labels
label_encoder = LabelEncoder()

y_train_xgb = label_encoder.fit_transform(y_train_uk)
y_test_xgb = label_encoder.transform(y_test_uk)

# One-hot encode categorical predictors
X_train_xgb = pd.get_dummies(
    X_train_uk,
    columns=categorical_features,
    drop_first=False
)

X_test_xgb = pd.get_dummies(
    X_test_uk,
    columns=categorical_features,
    drop_first=False
)

# Align test columns with training columns
X_test_xgb = X_test_xgb.reindex(
    columns=X_train_xgb.columns,
    fill_value=0
)

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(
    X_train_xgb,
    y_train_xgb
)

y_pred_xgb_encoded = xgb_model.predict(X_test_xgb)
y_pred_xgb_uk = label_encoder.inverse_transform(
    y_pred_xgb_encoded
)

print("XGBoost — UK Internal Validation")
print("-" * 50)

print(
    classification_report(
        y_test_uk,
        y_pred_xgb_uk,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

print(
    "Accuracy:",
    round(
        accuracy_score(y_test_uk, y_pred_xgb_uk),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_test_uk,
            y_pred_xgb_uk,
            average="macro",
            zero_division=0
        ),
        4
    )
)

In [ ]:
# Cell 41: Train XGBoost with the same alpha=0.75 power-smoothed class weights for a fair comparison with CatBoost.

xgb_alpha = 0.75

xgb_class_weights = {
    cls: weight ** xgb_alpha
    for cls, weight in class_weights.items()
}

sample_weights_xgb = (
    y_train_uk
    .map(xgb_class_weights)
    .values
)

print("XGBoost class weights:")
for cls, weight in xgb_class_weights.items():
    print(f"{cls}: {weight:.4f}")

xgb_weighted_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42
)

xgb_weighted_model.fit(
    X_train_xgb,
    y_train_xgb,
    sample_weight=sample_weights_xgb
)

y_pred_xgb_w_encoded = xgb_weighted_model.predict(
    X_test_xgb
)

y_pred_xgb_w = label_encoder.inverse_transform(
    y_pred_xgb_w_encoded
)

xgb_w_accuracy = accuracy_score(
    y_test_uk,
    y_pred_xgb_w
)

xgb_w_macro_f1 = f1_score(
    y_test_uk,
    y_pred_xgb_w,
    average="macro",
    zero_division=0
)

xgb_w_macro_recall = recall_score(
    y_test_uk,
    y_pred_xgb_w,
    average="macro",
    zero_division=0
)

print("\nWeighted XGBoost (alpha=0.75) — UK Internal Validation")
print("-" * 60)

print(
    classification_report(
        y_test_uk,
        y_pred_xgb_w,
        labels=["Fatal", "Serious", "Slight"],
        digits=4,
        zero_division=0
    )
)

print(f"Accuracy:     {xgb_w_accuracy:.4f}")
print(f"Macro Recall: {xgb_w_macro_recall:.4f}")
print(f"Macro F1:     {xgb_w_macro_f1:.4f}")

In [ ]:
# Cell 43: Compare all evaluated models on UK internal validation and identify the best overall severity classifier.

model_summary_uk = pd.DataFrame({
    "Model": [
        "Baseline CatBoost",
        "CatBoost alpha=0.75",
        "Fully Balanced CatBoost",
        "Controlled Resampling CatBoost",
        "Logistic Regression Balanced",
        "Baseline XGBoost",
        "Weighted XGBoost alpha=0.75"
    ],
    "Accuracy": [
        uk_accuracy,
        0.6634,
        uk_w_accuracy,
        resampled_accuracy,
        0.4826,
        0.7344,
        xgb_w_accuracy
    ],
    "Macro_F1": [
        uk_f1_macro,
        0.4014,
        uk_w_f1_macro,
        resampled_macro_f1,
        0.3330,
        0.3239,
        xgb_w_macro_f1
    ],
    "Macro_Recall": [
        uk_recall_macro,
        0.4216,
        uk_w_recall_macro,
        resampled_macro_recall,
        0.4569,
        0.3484,
        xgb_w_macro_recall
    ],
    "Fatal_Recall": [
        uk_class_report["Fatal"]["recall"],
        0.1649,
        0.4433,
        resampled_report["Fatal"]["recall"],
        0.5361,
        0.0034,
        0.1581
    ],
    "Serious_Recall": [
        uk_class_report["Serious"]["recall"],
        0.3069,
        0.4131,
        resampled_report["Serious"]["recall"],
        0.2872,
        0.0697,
        0.2904
    ]
}).round(4)

model_summary_uk = model_summary_uk.sort_values(
    "Macro_F1",
    ascending=False
)

print("UK Internal Validation — Final Model Comparison")
print(model_summary_uk.to_string(index=False))

In [ ]:
# Cell 44: Summarize the final selected CatBoost alpha=0.75 performance on UK internal validation and France external validation.

final_validation_summary = pd.DataFrame({
    "Validation_Set": [
        "UK Internal Validation",
        "France External Validation"
    ],
    "Accuracy": [
        0.6634,
        fr_final_accuracy
    ],
    "Macro_Precision": [
        0.4005,
        fr_final_precision_macro
    ],
    "Macro_Recall": [
        0.4216,
        fr_final_recall_macro
    ],
    "Macro_F1": [
        0.4014,
        fr_final_f1_macro
    ],
    "Fatal_Recall": [
        0.1649,
        0.1835
    ],
    "Serious_Recall": [
        0.3069,
        0.3720
    ],
    "Slight_Recall": [
        0.7930,
        0.6422
    ]
}).round(4)

print("Final Selected Model — Internal vs External Validation")
print(final_validation_summary.to_string(index=False))

print("\nGeneralization changes (France - UK):")
for metric in [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Fatal_Recall",
    "Serious_Recall",
    "Slight_Recall"
]:
    change = (
        final_validation_summary.loc[1, metric]
        - final_validation_summary.loc[0, metric]
    )
    print(f"{metric}: {change:+.4f}")

In [ ]:
# Cell 45: Save the final validation tables and collision-configuration analysis for reproducible reporting and manuscript preparation.

from pathlib import Path

output_dir = Path("../outputs/tables")
output_dir.mkdir(parents=True, exist_ok=True)

# Save final UK model comparison
model_summary_uk.to_csv(
    output_dir / "Table_Model_Comparison_UK_Internal.csv",
    index=False
)

# Save final internal vs external validation summary
final_validation_summary.to_csv(
    output_dir / "Table_Final_Internal_vs_External_Validation.csv",
    index=False
)

# Save France final confusion matrix
cm_france_final_df.to_csv(
    output_dir / "Table_France_Final_Confusion_Matrix.csv"
)

# Save collision configuration severity distributions
uk_config_pct.to_csv(
    output_dir / "Table_UK_Collision_Configuration_Severity.csv"
)

fr_config_pct.to_csv(
    output_dir / "Table_France_Collision_Configuration_Severity.csv"
)

print("Saved files:")
print("1. Table_Model_Comparison_UK_Internal.csv")
print("2. Table_Final_Internal_vs_External_Validation.csv")
print("3. Table_France_Final_Confusion_Matrix.csv")
print("4. Table_UK_Collision_Configuration_Severity.csv")
print("5. Table_France_Collision_Configuration_Severity.csv")

In [ ]:
# Cell 46: Save the final France confusion matrix and UK-vs-France performance comparison figures.

from pathlib import Path

figure_dir = Path("../outputs/figures")
figure_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# Figure 1: Final France confusion matrix
# --------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 6))

ConfusionMatrixDisplay(
    confusion_matrix=cm_france_final,
    display_labels=["Fatal", "Serious", "Slight"]
).plot(
    ax=ax,
    values_format="d",
    cmap="Blues",
    colorbar=False
)

ax.set_title(
    "Final Cost-Sensitive CatBoost — France External Validation"
)

plt.tight_layout()

plt.savefig(
    figure_dir / "Figure_France_Final_Confusion_Matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# --------------------------------------------------
# Figure 2: UK internal vs France external validation
# --------------------------------------------------

metrics_to_plot = [
    "Accuracy",
    "Macro_Precision",
    "Macro_Recall",
    "Macro_F1",
    "Fatal_Recall",
    "Serious_Recall",
    "Slight_Recall"
]

plot_df = final_validation_summary.set_index(
    "Validation_Set"
)[metrics_to_plot].T

ax = plot_df.plot(
    kind="bar",
    figsize=(10, 6)
)

ax.set_title(
    "Final Model Performance: UK Internal vs France External Validation"
)

ax.set_ylabel("Score")
ax.set_xlabel("Metric")
ax.set_ylim(0, 1)

plt.xticks(rotation=35, ha="right")
plt.legend(title="Validation Set")

plt.tight_layout()

plt.savefig(
    figure_dir / "Figure_UK_vs_France_Final_Validation.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("Saved figures:")
print("1. Figure_France_Final_Confusion_Matrix.png")
print("2. Figure_UK_vs_France_Final_Validation.png")